This is the reproduction notebook for the main PCSS + MiSS experiment for `Qwen3.5-9B-Base`.

This specific notebook was tested on this docker image on a H100 SXM: `vastai/pytorch:2.10.0-cuda-13.0.2-py312-24.04-2026-03-26`

It takes about ~40 minutes to train on a H100 SXM / H200 NVL. You can use other hardware for sure, just make sure to switch `attn_implementation` accordingly (e.g., to 'sdpa')

If you are using the same docker image and hopper hardware (with cu130 compatibility), you can just run all cells. When you're connecting to the jupyter server, select `main venv`

We use a custom transformers fork for dense Qwen 3.5 MTP training support: https://github.com/tamewild/transformers/tree/verify-qwen-mtp-implementation

##### Installation

Here we install `uv` and prerequisites:

In [2]:
%pip install uv
!/venv/main/bin/python -m uv pip install "git+https://github.com/tamewild/transformers.git@7ccc0ddb811422a86c7737a5497e05352a1c9ae3" trl==1.4.0 peft==0.19.1 bitsandbytes==0.49.2 kernels==0.14.1
!/venv/main/bin/python -m uv pip install git+https://github.com/linkedin/Liger-Kernel.git@30b8486a2bd48dff97f22c5f0c88520b8825cb35
!/venv/main/bin/python -m uv pip install flash-linear-attention==0.5.0
!/venv/main/bin/python -m uv pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu13torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!/venv/main/bin/python -m uv pip install tilelang==0.1.11 apache-tvm-ffi==0.1.11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 96.8 MB/s  0:00:006m0:00:01
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.13 environment at: /venv/main
Resolved 79 packages in 20.58s                                       
Prepared 31 packages in 3.22s                                            
Uninstalled 3 packages in 149ms
Installed 31 packages in 1.01sfrom git+https://github.com/ta
 + accelerate==1.14.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + attrs==26.1.0
 + bitsandbytes==0.49.2
 + charset-normalizer==3.5.1
 - click==8.3.1
 + click==8.5.0
 + datasets==5.0.1
 + dill==0.4.1
 + frozenlist==1.8.0
 - hf-xet==1.4.2
 + hf-xet==1.6.0
 - huggingface-hub==1.8.0
 + huggingface-hub==1.30.0
 + kernels==0.14.1
 + kernels-data==0.16.1
 + multidict==6.7.1
 + multiprocess==0.70.19
 + pandas==3.0.5
 + peft==0.19.1
 + propcache==0.5.2
 + pyarrow==25.0.1
 + regex==2026.9.3
 + requests==2.34.2
 + safetensors==0.8.0
 + tokenize

In case that something goes wrong, make sure `%pip freeze` matches the following output:

In [3]:
%pip freeze

accelerate==1.14.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiosignal==1.4.0
annotated-doc==0.0.4
anyio==4.13.0
apache-tvm-ffi==0.1.11
asttokens==3.0.1
attrs==26.1.0
bitsandbytes==0.49.2
causal-conv1d @ https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu13torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
certifi==2026.2.25
charset-normalizer==3.5.1
click==8.5.0
cloudpickle==3.1.2
comm==0.2.3
cuda-bindings==13.0.3
cuda-pathfinder==1.5.0
datasets==5.0.1
debugpy==1.8.20
decorator==5.2.1
dill==0.4.1
einops==0.8.2
executing==2.2.1
filelock==3.25.2
fla-core==0.5.0
flash-linear-attention==0.5.0
frozenlist==1.8.0
fsspec==2026.2.0
h11==0.16.0
hf-xet==1.6.0
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.30.0
idna==3.11
ipykernel==7.2.0
ipython==9.11.0
ipython_pygments_lexers==1.1.1
ipywidgets==8.1.8
jedi==0.19.2
Jinja2==3.1.6
jupyter_client==8.8.0
jupyter_core==5.9.1
jupyterlab_widgets==3.0.16
kernels==0.14.1
kernels-data==0.16.1
liger_kernel 

In [4]:
import os, torch, copy
from transformers import AutoModelForMultimodalLM, AutoTokenizer
from liger_kernel.transformers import apply_liger_kernel_to_qwen3_5
from typing import Optional, List
import torch
import torch.nn as nn

apply_liger_kernel_to_qwen3_5() # apply here so we benefit during inference

model = AutoModelForMultimodalLM.from_pretrained(
    "Qwen/Qwen3.5-9B-Base",
    attn_implementation = "kernels-community/flash-attn3@v1",
    dtype = torch.bfloat16,
    device_map = "cuda"
)
model.config.text_config.use_cache = False
# use unsloth fixed official chat template
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-9B")

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.99k [00:00<?, ?B/s]

##### Dataset Processing

In [5]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from typing import Union, List, Tuple

def create_pcss_mtp_dataset(
    dataset: Union[Dataset, DatasetDict],
    tokenizer: AutoTokenizer,
    ref_model: nn.Module
) -> Union[Dataset, DatasetDict]:
    ref_model.eval()
    
    def _process_single_example(example: dict) -> dict:
        prompt_str = tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": example["prompt"]
                }
            ],
            add_generation_prompt=True,
            enable_thinking=False,
            tokenize=False
        )
        full_str = prompt_str + example["completion"] + tokenizer.eos_token

        prompt_tokens = tokenizer(prompt_str, return_tensors="pt")
        full_tokens = tokenizer(full_str, return_tensors="pt")
        prompt_len = prompt_tokens.input_ids.shape[1]

        labels = full_tokens.input_ids.clone()
        labels[:, :prompt_len] = -100

        inputs_for_ref = {
            "input_ids": full_tokens.input_ids.to(ref_model.device),
            "attention_mask": full_tokens.attention_mask.to(ref_model.device),
            "labels": labels.to(ref_model.device),
            "num_mtp_steps": 1
        }

        with torch.no_grad():
            outputs = ref_model.forward_mtp(**inputs_for_ref)
            losses = [outputs.loss_0] + outputs.mtp_losses
            l_refs = [l.cpu().item() for l in losses]

        return {
            "input_ids": full_tokens.input_ids.squeeze(0).tolist(),
            "attention_mask": full_tokens.attention_mask.squeeze(0).tolist(),
            "labels": labels.squeeze(0).tolist(),
            "l_refs": l_refs,
        }

    processed_dataset = None
    if isinstance(dataset, DatasetDict):
        processed_splits = {}
        for split_name, split_dataset in dataset.items():
            processed_examples = [
                _process_single_example(ex)
                for ex in tqdm(split_dataset, desc=f"Processing split '{split_name}'")
            ]
            processed_splits[split_name] = Dataset.from_list(processed_examples)
        processed_dataset = DatasetDict(processed_splits)
    elif isinstance(dataset, Dataset):
        processed_examples = [
            _process_single_example(ex)
            for ex in tqdm(dataset, desc="Processing dataset")
        ]
        processed_dataset = Dataset.from_list(processed_examples)
    else:
        raise TypeError(f"Input must be a Dataset or DatasetDict, but got {type(dataset)}")

    return processed_dataset

dataset = load_dataset("tamewild/instruct5")

dataset = dataset.map(lambda example: {
    "prompt": example["conversation"][0]["content"],
    "completion": example["conversation"][1]["content"]
})

dataset = create_pcss_mtp_dataset(
    dataset,
    tokenizer,
    model
)

README.md:   0%|          | 0.00/472 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.61MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  718kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

Processing split 'train':   0%|          | 0/500 [00:00<?, ?it/s]

Processing split 'test':   0%|          | 0/77 [00:00<?, ?it/s]

##### Dataset Inspection

I recommend using HF's dataset inspector on the website, it's convenient. But here you can inspect the first entry.

See first entry without prompt masked:

In [6]:
print(tokenizer.decode(dataset['train'][0]['input_ids']))

<|im_start|>user
There are 5 houses in a row, numbered from 1 through 5 from left to right. Each house is home to one person and each resident has a unique attribute from the following characteristics:

Favorite Bird: Owl, Falcon, Penguin, Eagle, Robin
Preferred Furniture Style: Traditional, Industrial, Rustic, Bohemian, Mid-Century Modern
Favorite Smoothie: Peanut Butter Banana, Green Smoothie, Tropical Twist, Strawberry Banana, Berry Blast
Favorite Day of the Week: Monday, Sunday, Thursday, Wednesday, Friday
Favorite Vacation Destination: National Parks, Caribbean Cruise, Camping, Hawaii, Italy

1. The Sunday enthusiast is somewhere to the left of the person whose favorite smoothie is Tropical Twist.
2. The person who loves robins is next to the person who dreams of Hawaii.
3. The cruise lover is somewhere to the left of the eagle admirer.
4. There are 2 houses between the person who loves robins and the falcon admirer.
5. The national parks explorer is somewhere to the left of the p

See first entry with prompt masked:

In [7]:
def print_unmasked_part(entry):
    unmasked = list(filter(lambda label: label != -100, entry["labels"]))
    print(tokenizer.decode(unmasked))

print_unmasked_part(dataset['train'][0])

We are given a complex logic puzzle with 5 houses, each having a unique value for:

- **Bird**: Owl, Falcon, Penguin, Eagle, Robin  
- **Furniture**: Traditional, Industrial, Rustic, Bohemian, Mid-Century Modern  
- **Smoothie**: Peanut Butter Banana, Green Smoothie, Tropical Twist, Strawberry Banana, Berry Blast  
- **Day**: Monday, Sunday, Thursday, Wednesday, Friday  
- **Vacation**: National Parks, Caribbean Cruise, Camping, Hawaii, Italy  

We are to assign one of each attribute to each house (numbered 1 to 5 from left to right), satisfying all 21 clues.

We'll solve this step by step using deduction.

---

### Step 1: Set up the grid

We’ll create a table with 5 columns (House 1 to House 5), and 5 rows (Bird, Furniture, Smoothie, Day, Vacation).

We’ll fill in step by step.

---

### Step 2: Parse the clues

We'll go through each clue and extract direct or indirect information.

---

**Clue 1**: The Sunday enthusiast is somewhere to the left of the person whose favorite smoothie 

##### Training

In [8]:
from peft import get_peft_model, MissConfig

model = get_peft_model(
    model,
    MissConfig(
        r = 512,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
            "in_proj_a", "in_proj_b", "in_proj_qkv", "in_proj_z", "out_proj",
            "fc"
        ],
        bias = "none",
        modules_to_save = None,
        task_type="CAUSAL_LM",
        init_weights = True # "MiSS efficience and balance"
    ),
    autocast_adapter_dtype = False # stay in bf16
)

model.print_trainable_parameters()

trainable params: 754,712,576 || all params: 10,407,816,944 || trainable%: 7.2514


In [9]:
from transformers import Trainer
import torch
import torch.nn as nn
from typing import Dict, Union, Any, Optional

class PCSSTrainer(Trainer):
    def __init__(self, *args, beta, peak_scale, global_l_std, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        self.peak_scale = peak_scale
        self.global_l_std = global_l_std

    def compute_loss(
        self,
        model: nn.Module,
        inputs: Dict[str, Union[torch.Tensor, Any]],
        return_outputs: bool = False,
        num_items_in_batch: Optional[torch.Tensor] = None,
    ):
        l_refs = inputs.pop("l_refs").view(-1)

        mtp_outputs = model.forward_mtp(**inputs, num_mtp_steps=1)

        mode = "train" if self.model.training else "eval"

        if mode == "train":
            # Base Model (Step 0)
            l_sft_0 = mtp_outputs.loss_0
            l_ref_0 = l_refs[0]
            std_0 = self.global_l_std[0]

            with torch.no_grad():
                advantage_0 = (l_ref_0 - l_sft_0) / std_0
                sigmoid_input_0 = self.beta * advantage_0
                sigmoid_val_0 = torch.sigmoid(sigmoid_input_0)
                sigmoid_prime_0 = sigmoid_val_0 * (1.0 - sigmoid_val_0)
                scale_0 = self.peak_scale * (4.0 * sigmoid_prime_0)

            scaled_loss_0 = l_sft_0 * scale_0.detach()

            # MTP Step 1
            l_sft_1 = mtp_outputs.mtp_losses[0]
            l_ref_1 = l_refs[1]
            std_1 = self.global_l_std[1]

            with torch.no_grad():
                advantage_1 = (l_ref_1 - l_sft_1) / std_1
                sigmoid_input_1 = self.beta * advantage_1
                sigmoid_val_1 = torch.sigmoid(sigmoid_input_1)
                sigmoid_prime_1 = sigmoid_val_1 * (1.0 - sigmoid_val_1)
                scale_1 = self.peak_scale * (4.0 * sigmoid_prime_1)

            scaled_loss_1 = l_sft_1 * scale_1.detach()

            final_loss = 0.5 * (scaled_loss_0 + scaled_loss_1)

            return (final_loss, mtp_outputs) if return_outputs else final_loss
        else:
            return (mtp_outputs.loss_0, mtp_outputs) if return_outputs else mtp_outputs.loss_0

In [10]:
from transformers import TrainingArguments, default_data_collator

def std_calc(l_refs):
    tensor_refs = torch.tensor(l_refs)
    return torch.clamp(tensor_refs.std(dim=0), min=1e-6).tolist()

global_l_std = std_calc(dataset['train']['l_refs'])

print(f'Global Standard Deviations: {global_l_std}')

trainer = PCSSTrainer(
    model = model,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    data_collator = default_data_collator,
    beta=0.65,
    peak_scale=5,
    global_l_std=global_l_std,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 500, # fixed at 1 epoch
        num_train_epochs = 5,
        learning_rate = 5e-6,
        fp16 = False,
        bf16 = True,
        logging_strategy = "no",
        optim = "adamw_8bit",
        adam_beta2 = 0.99994, # scale optimizer memory. according to paper
        weight_decay = 0.01,
        lr_scheduler_type = "constant_with_warmup",
        seed = 3407,
        output_dir = "workspace",
        save_strategy = "steps",
        save_steps = 2500,
        eval_strategy = "epoch",
        gradient_checkpointing = False, # highly recommended for h100, h200, etc
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        use_liger_kernel = False,
        prediction_loss_only = True,
        remove_unused_columns = False
    )
)

Global Standard Deviations: [0.027728069573640823, 0.03913002088665962]


In [11]:
trainer.train()

2026-09-08 23:59:24  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:130): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-09-08 23:59:36  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:138): TileLang completes to compile kernel `kernel`


Epoch,Training Loss,Validation Loss
1,No log,0.220072
2,No log,0.213969
3,No log,0.217550
4,No log,0.227000
5,No log,0.233485


TrainOutput(global_step=2500, training_loss=0.2686085205078125, metrics={'train_runtime': 2399.6805, 'train_samples_per_second': 1.042, 'train_steps_per_second': 1.042, 'total_flos': 1.187634414363644e+18, 'train_loss': 0.2686085205078125, 'epoch': 5.0})

In [12]:
import gc
from peft import PeftModel

# free vram
del model, trainer
torch.cuda.empty_cache()
gc.collect()

# merge with `autocast_adapter_dtype=True` (fp32)
model = AutoModelForMultimodalLM.from_pretrained(
    "Qwen/Qwen3.5-9B-Base",
    attn_implementation = "sdpa",
    dtype = torch.bfloat16,
    device_map = "cuda"
)
model = PeftModel.from_pretrained(
    model,
    "/workspace/checkpoint-2500",
    autocast_adapter_dtype = True
)
model = model.merge_and_unload()

# save locally, you can optionally upload to HF (look it up) but please avoid polluting HF with essentially the same model trained in a few minutes
model.save_pretrained("./merged")
tokenizer.save_pretrained("./merged")

# free vram again
del model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/775 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

##### AIME 2025 Evaluation using vLLM

In [13]:
# NOTE: This is for cu130, make sure your device supports it. Also pin fastapi because of https://github.com/vllm-project/vllm/issues/45597
!/venv/main/bin/python -m uv pip install vllm==0.19.1 fastapi==0.136.3 transformers==5.8.1 --extra-index-url https://wheels.vllm.ai/0.19.1/cu130
!/venv/main/bin/python -m uv pip install math-verify[antlr4_13_2]==0.9.0 #pinned version of hf math-verify

Using Python 3.12.13 environment at: /venv/main
Resolved 180 packages in 3.29s                                       
Prepared 107 packages in 4.68s                                           
Uninstalled 2 packages in 1.57s
Installed 107 packages in 7.21s6.6                          
 + agent-detector==2.0.0
 + annotated-types==0.8.0
 + anthropic==1.4.0
 + astor==0.8.1
 + blake3==1.0.9
 + cachetools==7.1.8
 + cbor2==6.1.4
 + cffi==2.1.1
 + compressed-tensors==0.15.0.1
 + cryptography==50.0.1
 + cuda-python==13.0.3
 + depyf==0.20.0
 + detect-installer==0.2.1
 + diskcache==5.6.3
 + distro==1.9.0
 + dnspython==2.8.0
 + docstring-parser==0.18.0
 + email-validator==2.3.0
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.24.0
 + fastar==0.12.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + gguf==0.19.0
 + googleapis-common-protos==1.75.3
 + grpcio==1.83.1
 + httpcore2==2.3.0
 + httptools==0.8.0
 + httpx-sse==0.4.3
 + httpx2==2.3.0
 + ijson==3.5.1
 + interegular==0

**NOTE**: You may have to restart kernel before running the following cell

In [1]:
from datasets import load_dataset
from vllm import LLM, SamplingParams
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

llm = LLM(
    model = "./merged",
    max_model_len = 17_000,
    gpu_memory_utilization = 0.85,
    language_model_only = True,
    generation_config = "vllm",
    disable_log_stats = False,
    speculative_config = {
        "method": "mtp",
        "num_speculative_tokens": 2,
    }
)

def extract_last_box(s: str) -> str:
    tag = "\\boxed{"
    tag_start = s.rfind(tag)
    if tag_start == -1:
        return ""

    content = s[tag_start + len(tag):]
    depth = 1
    for i, char in enumerate(content):
        if char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                return content[:i].strip()

    return ""

def check_math500(ground_truth, response) -> bool:
    boxed_content = extract_last_box(response)
    if not boxed_content:
        return False

    boxed_response = f"\\boxed{{{boxed_content}}}"

    # Ensure ground truth is parsable by math_verify
    if "\\boxed" not in ground_truth:
        ground_truth = f"\\boxed{{{ground_truth}}}"

    gold = parse(
        ground_truth,
        # https://github.com/huggingface/Math-Verify/blob/ba3d3aaff23b3f4cac7a14672b4f6e293d97c98b/src/math_verify/tasks.py#L219
        [LatexExtractionConfig(boxed_match_priority=0)]
    )
    resp = parse(
        boxed_response,
        # https://github.com/huggingface/Math-Verify/blob/ba3d3aaff23b3f4cac7a14672b4f6e293d97c98b/src/math_verify/tasks.py#L221
        [
            LatexExtractionConfig(boxed_match_priority=0),
            ExprExtractionConfig()
        ]
    )
    return verify(gold, resp)

def benchmark_aime2025(llm):
    aime2025_benchmark = load_dataset("MathArena/aime_2025")['train']

    extra = r"Please reason step by step, and put your final answer within \boxed{}"

    aime2025_benchmark = aime2025_benchmark.map(lambda example: {
        "problem": example["problem"].strip() + f"\n\n{extra}"
    })

    prompts = [[{"role": "user", "content": question["problem"]}] for question in aime2025_benchmark]

    prompt_outputs = llm.chat(
        prompts,
        SamplingParams(
            n=6,
            temperature=1.0,
            top_p=0.95,
            top_k=20,
            max_tokens=16_384
        ),
        chat_template_kwargs={"enable_thinking": False},
        use_tqdm=True
    )

    correct = 0

    for solution, prompt_output in zip(aime2025_benchmark['answer'], prompt_outputs):
        solution = str(solution)
        for output in prompt_output.outputs:
            if check_math500(solution, output.text):
                correct += 1

    print(f"Correct answers: {correct}")
    print(f"Estimated pass@1: {correct / (len(aime2025_benchmark['problem']) * 6)}")

benchmark_aime2025(llm)

INFO 09-09 00:40:44 [utils.py:233] non-default args: {'max_model_len': 17000, 'gpu_memory_utilization': 0.85, 'language_model_only': True, 'speculative_config': {'method': 'mtp', 'num_speculative_tokens': 2}, 'generation_config': 'vllm', 'model': './merged'}
INFO 09-09 00:40:44 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 09-09 00:40:44 [model.py:1678] Using max model len 17000
INFO 09-09 00:40:44 [model.py:549] Resolved architecture: Qwen3_5MTP
INFO 09-09 00:40:44 [model.py:1678] Using max model len 262144
WARNING 09-09 00:40:44 [speculative.py:512] Enabling num_speculative_tokens > 1 will run multiple times of forward on same MTP layer,which may result in lower acceptance rate
INFO 09-09 00:40:44 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 09-09 00:40:45 [config.py:281] Setting attention block size to 544 tokens to ensure that attention page size is >= mamba page size.
INFO 09-09 00:40:45 [config.py:312] Padding mamba page size by 2.26% to ensure that mamba page size and attention page size are exactly equal.
INFO 09-09 00:40:45 [vllm.py:790] Asynchronous scheduling is enabled.
INFO 09-09 00:40:47 [registry.py:126] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
WARNING 09-09 00:40:48 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=6078) INFO 09-09 00:40:53 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='./merged', speculative_config=SpeculativeConfig(method='mtp', model='./merged', num_spec_tokens=2), tokenizer=

(EngineCore pid=6078) [transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


(EngineCore pid=6078) INFO 09-09 00:40:56 [registry.py:126] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=6078) INFO 09-09 00:40:56 [parallel_state.py:1400] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.17.0.3:42079 backend=nccl
(EngineCore pid=6078) INFO 09-09 00:40:56 [parallel_state.py:1716] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=6078) WARNING 09-09 00:40:56 [__init__.py:206] min_p and logit_bias parameters won't work with speculative decoding.
(EngineCore pid=6078) INFO 09-09 00:40:57 [gpu_model_runner.py:4735] Starting to load model ./merged...
(EngineCore pid=6078) INFO 09-09 00:40:57 [cuda.py:390] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=6078) INFO 09-09 00:40:57 [mm_encoder_attention.py:230] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore 

...

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Rendering conversations:   0%|          | 0/30 [00:00<?, ?it/s]

INFO 09-09 00:43:24 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/180 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

INFO 09-09 00:43:42 [loggers.py:259] Engine 000: Avg prompt throughput: 1938.5 tokens/s, Avg generation throughput: 5116.3 tokens/s, Running: 180 reqs, Waiting: 0 reqs, GPU KV cache usage: 88.3%, Prefix cache hit rate: 0.0%
INFO 09-09 00:43:42 [metrics.py:101] SpecDecoding metrics: Mean acceptance length: 2.52, Accepted throughput: 3077.27 tokens/s, Drafted throughput: 4059.39 tokens/s, Accepted: 63098 tokens, Drafted: 83236 tokens, Per-position acceptance rate: 0.892, 0.624, Avg Draft acceptance rate: 75.8%
INFO 09-09 00:43:52 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 11094.7 tokens/s, Running: 174 reqs, Waiting: 0 reqs, GPU KV cache usage: 93.0%, Prefix cache hit rate: 0.0%
INFO 09-09 00:43:52 [metrics.py:101] SpecDecoding metrics: Mean acceptance length: 2.53, Accepted throughput: 6704.29 tokens/s, Drafted throughput: 8787.15 tokens/s, Accepted: 67176 tokens, Drafted: 88046 tokens, Per-position acceptance rate: 0.898, 0.628, Avg Dra

Training is **not** deterministic and vLLM itself has non-determinism even with greedy decoding.

Due to longer training times, we only ran it twice (one for the ablation matrix and one for reproduction).

We got 60.67% and ~58% respectively

You can expect your reproduced score to land somewhere in this general neighborhood, which is well above the 49.44% base model baseline.